In [1]:
import scipy

from src.utils import ingaas_processing
import numpy as np
import matplotlib.pyplot as plt

In [2]:
scan_path = "C://Users//jeppe//Documents//LIPI//scans//outside2//scan_20260409_142211.raw"
#scan_path = "C://Users//jeppe//Documents//LIPI//scans//outside2//scan_20260409_135820.raw"
cal_path = "data/calibration.npz"
K,P,DIM = ingaas_processing.load_calibration(cal_path)

In [7]:
#Functions
def get_phase(x, f, f_s, debug_out=False):
    '''
    Finds the phase of a given frequency in a signal, sampled at some other frequency.

    Args:
        x: The signal as a numpy array
        f: The frequency to find the phase of
        f_s: The frequency at which the signal was sampled
        debug_out: Whether to display some debug info

    Returns:
        phase: The phase of the signal
        f_out: The closest frequency present in the FFT

    '''

    # Where is the modulation frequency in the results?
    fft_frequencies = scipy.fft.fftshift(scipy.fft.rfftfreq(len(x), d=1/f_s))
    f_index = np.argmin(np.abs(f-fft_frequencies))
    f_out = fft_frequencies[f_index]
    if debug_out:
        print(f_index)
        print(f_out)

    # FFT of data
    yf = scipy.fft.fftshift(scipy.fft.rfft(x))
    if debug_out:
        plt.plot(fft_frequencies, yf)
        plt.show()


    # Get phase
    phase = np.angle(yf[f_index])
    if debug_out:
        print(yf[f_index])
        print("Phase:", phase)

    return phase, f_out

def get_extrema(f,phase,f_s,N):
    '''
    Estimate the extrema points of a discrete signal, specifically finding peak and valley pairs

    Args:
        f: Frequency of the signal component of interest
        phase: How the signal component is shifted. Should be in the range [-Pi,Pi]
        f_s: Frequency at which the signal is sampled
        N: Number of peaks and valley pairs to find

    Returns:
        peaks: Index of peaks as a numpy array
        valleys: Index of valleys as a numpy array

    '''
    peak_arg =  -phase

    if peak_arg < 0:
        peak_arg += 2*np.pi

    if peak_arg > 2*np.pi:
        peak_arg -= 2*np.pi

    valley_arg = peak_arg + np.pi

    base = np.arange(N)*2*np.pi
    peaks = np.round(f_s*(base+peak_arg)/(2*np.pi*f)).astype(np.int32)
    valleys = np.round(f_s*(base+valley_arg)/(2*np.pi*f)).astype(np.int32)

    #mask = np.all([peaks < N, valleys < N], axis=0)
    #return peaks[mask], valleys[mask]
    return peaks, valleys

def get_window(file)

In [4]:
#Parameters
f_m = 50 #Hz
#f_s = 300 #Hz
f_s = 299.917668 #Hz

start = 0
wlen = 600

N = len(signal)

min_steps = np.ceil(N/wlen).astype(np.int32)
steps = 10+min_steps
step_size = (N-start-wlen)/(steps-1)
#print(step_size)

peaks = np.empty(0, dtype=np.int32)
valleys = np.empty(0, dtype=np.int32)
for i in range(steps):
    w_start = round(i*step_size)+start
    w_end = w_start+wlen
    w_start_next = round((i+1)*step_size)+start
    #print(w_start, w_end, w_start_next)
    if w_end+10 > N:
        w_start_next = w_end

    window = signal[w_start:w_end]
    phase, f_out = get_phase(window, f_m, f_s)
    peaks_i,valleys_i = get_extrema(f_m, phase, f_s, w_start_next-w_start)
    #print(peaks_i,valleys_i)
    peaks = np.append(peaks, peaks_i+w_start)
    valleys = np.append(valleys, valleys_i+w_start)

plt.plot(signal)
plt.plot(peaks, signal[peaks], ".g")
plt.plot(valleys, signal[valleys], ".r")
plt.show()

print(np.unique(peaks-valleys, return_counts=True))
frame_profile = np.sum(scan, axis=2)
frame_diffs = frame_profile[1:]-frame_profile[:-1]
#plt.plot(np.mean(np.abs(frame_diffs), axis=0))
lom = np.argmax(np.mean(np.abs(frame_diffs), axis=0))
print(lom)
result = data[peaks,:,:] - data[valleys,:,:]
stitch = result[:,lom,:]

NameError: name 'signal' is not defined